# Manejo de Errores, Behaviours y Protocolos en Elixir

Este notebook explora cómo Elixir maneja situaciones excepcionales de forma idiomática y cómo define interfaces y polimorfismo mediante Behaviours y Protocolos.

## 1. Manejo de Errores Idiomático

A diferencia de otros lenguajes, en Elixir no se suelen usar excepciones para el control de flujo. La convención es retornar tuplas `{:ok, valor}` o `{:error, razon}`.

In [ ]:
defmodule Validador do
  def revisar_edad(edad) when edad >= 18, do: {:ok, "Acceso permitido"}
  def revisar_edad(_), do: {:error, "Eres menor de edad"}
end

case Validador.revisar_edad(20) do
  {:ok, msg} -> IO.puts("Éxito: #{msg}")
  {:error, msg} -> IO.puts("Fallo: #{msg}")
end

### El operador `with` 

Se utiliza para encadenar múltiples operaciones que retornan tuplas de éxito/error, deteniéndose en el primer error encontrado.

In [ ]:
usuarios = %{1 => %{nombre: "Maxi", activo: true}}

defmodule App do
  def obtener_perfil(id, db) do
    with {:ok, usuario} <- Map.fetch(db, id),
         true <- usuario.activo do
      {:ok, usuario}
    else
      :error -> {:error, "Usuario no encontrado"}
      false -> {:error, "Cuenta inactiva"}
    end
  end
end

IO.inspect App.obtener_perfil(1, usuarios)
IO.inspect App.obtener_perfil(2, usuarios)

## 2. Excepciones (`try`, `rescue`, `raise`)

Se reservan para errores que el programador no espera manejar durante el flujo normal (errores catastróficos o de lógica).

In [ ]:
try do
  raise "¡Algo salió muy mal!"
rescue
  e in RuntimeError -> IO.puts("Error capturado: #{e.message}")
after
  IO.puts("Limpieza de recursos...")
end

## 3. Behaviours (Comportamientos)

Los Behaviours permiten definir una interfaz (un conjunto de funciones) que un módulo debe implementar. Son similares a las interfaces en Java o Traits en Rust.

In [ ]:
defmodule Trabajador do
  @callback ejecutar(datos :: any()) :: {:ok, any()} | {:error, String.t()}
end

defmodule MiImplementacion do
  @behaviour Trabajador

  @impl Trabajador
  def ejecutar(datos) do
    {:ok, "Procesando #{datos}"}
  end
end

MiImplementacion.ejecutar("archivos")

## 4. Protocolos (Polimorfismo)

Los Protocolos permiten despachar funciones basándose en el tipo del primer argumento. Es la forma en que Elixir logra el polimorfismo.

In [ ]:
defprotocol Utilidad do
  def duplicar(dato)
end

defimpl Utilidad, for: Integer do
  def duplicar(n), do: n * 2
end

defimpl Utilidad, for: String do
  def duplicar(s), do: s <> s
end

defimpl Utilidad, for: List do
  def duplicar(l), do: l ++ l
end

IO.puts Utilidad.duplicar(5)
IO.puts Utilidad.duplicar("Ho")
IO.inspect Utilidad.duplicar([1, 2])

## 5. Resumen de Diferencias

| Concepto | Propósito |
| :--- | :--- |
| **Tuplas {:ok, _}** | Manejo de errores esperado y flujo normal. |
| **Excepciones** | Errores inesperados o irrecuperables. |
| **Behaviours** | Definir un contrato para módulos (basado en módulos). |
| **Protocolos** | Implementar la misma funcionalidad para diferentes tipos de datos. |